In [2]:
# Dependencias y utilidades compartidas
from itertools import product
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = Path("..").resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from optimization.differential_evolution import run_differential_evolution
from optimization.evolutionary import run_evolutionary
from optimization.gradient_descent_rosenbrock import run_gradient_descent_rosenbrock
from optimization.pso import run_pso
from optimization.rastrigin import rastrigin
from optimization.rosenbrock import rosenbrock



from tsp.ant_colony import run_ant_colony_tsp
from tsp.cost_model import compute_total_cost_matrix
from tsp.genetic_tsp import run_genetic_tsp


# Optimizacion de hiperparametros

La busqueda se realiza sobre Rosenbrock y Rastrigin en 3D. Para esta fase exploratoria se recomienda un presupuesto de **10000 evaluaciones de f por configuracion**, suficiente para distinguir tendencias sin hacer que la grilla sea excesivamente costosa.

El presupuesto se mantiene fijo y no se optimiza:

- PSO: 20 particulas y 499 iteraciones -> 20 * (499 + 1) = 10000.
- Evolutivo: 20 individuos, 10 hijos por generacion y 998 generaciones -> 20 + 10 * 998 = 10000.
- DE: 20 individuos y 499 generaciones -> 20 * (499 + 1) = 10000.

Se usan tres semillas (42, 43 y 44). La configuracion principal de produccion conserva su presupuesto de 40000 evaluaciones en configs/part1.yaml; este cuaderno usa 10000 solo para explorar hiperparametros.

In [3]:
# Funcion para la busqueda en cuadrícula de hiperparametros
def grid_search(runner, parameter_grid, seeds=(42, 43, 44), threshold=1e-4):
    keys = tuple(parameter_grid)
    rows = []
    for values in product(*(parameter_grid[key] for key in keys)):
        parameters = dict(zip(keys, values))
        results = [runner(parameters, seed) for seed in seeds]
        scores = np.asarray([result["best_value"] for result in results], dtype=float)
        rows.append({**parameters, "mean_best_value": float(np.nanmean(scores)),
                     "std_best_value": float(np.nanstd(scores)),
                     "success_rate": float(np.mean(scores <= threshold))})
    return sorted(rows, key=lambda row: (row["mean_best_value"], -row["success_rate"]))


def show_top(rows, count=5):
    for rank, row in enumerate(rows[:count], start=1):
        print(f"{rank}. {row}")


def make_population_runner(algorithm, objective, fixed_parameters, dimension=3, domain=(-5.12, 5.12)):
    bounds = [domain] * dimension
    def runner(parameters, seed):
        return algorithm(objective, bounds, seed=seed, **fixed_parameters, **parameters)
    return runner

print(f"Proyecto: {ROOT}")
print("Presupuesto exploratorio fijo: 10000 evaluaciones de f por configuracion")

Proyecto: /Users/nicoruizbl/Documents/GitHub/Aplicaciones-de-Optimizaci-n-con-Algoritmos-Bioinspirados
Presupuesto exploratorio fijo: 10000 evaluaciones de f por configuracion


# Optimizacion de descenso por gradiente

Se optimiza GD sobre Rosenbrock en 3D. Las iteraciones permanecen fijas en 10000; solo se explora la tasa de aprendizaje.

- learning_rate: [1e-4, 5e-4, 1e-3, 2e-3, 5e-3].
- iterations: 10000, fijo.

In [29]:
# Busqueda de grilla para GD en Rosenbrock 3D
def gd_rosenbrock_runner(parameters, seed):
    rng = np.random.default_rng(seed)
    initial = rng.uniform(-2, 2, size=3)
    return run_gradient_descent_rosenbrock(initial, **parameters)

gd_grid = {
    "learning_rate": [1e-4, 5e-4, 1e-3, 2e-3, 5e-3],
    "iterations": [2500],
}
gd_rosenbrock_results = grid_search(gd_rosenbrock_runner, gd_grid)
show_top(gd_rosenbrock_results)

1. {'learning_rate': 0.0005, 'iterations': 2500, 'mean_best_value': 0.09237723828844487, 'std_best_value': 0.022986174088977532, 'success_rate': 0.0}
2. {'learning_rate': 0.0001, 'iterations': 2500, 'mean_best_value': 0.941796147104649, 'std_best_value': 0.5728778487224627, 'success_rate': 0.0}
3. {'learning_rate': 0.001, 'iterations': 2500, 'mean_best_value': inf, 'std_best_value': nan, 'success_rate': 0.0}
4. {'learning_rate': 0.002, 'iterations': 2500, 'mean_best_value': inf, 'std_best_value': nan, 'success_rate': 0.0}
5. {'learning_rate': 0.005, 'iterations': 2500, 'mean_best_value': inf, 'std_best_value': nan, 'success_rate': 0.0}


# Optimizacion de PSO

El presupuesto se mantiene fijo: 40 particulas y 999 iteraciones. Solo se exploran inercia, componente cognitiva y componente social.

- inertia: 0.4, 0.7 o 0.9.
- cognitive y social: 1.0, 1.4 o 2.0.

In [30]:
# Busqueda de grilla para PSO en Rastrigin 3D
pso_fixed = {"particles": 20, "iterations": 499}
pso_grid = {
    "inertia": [0.4, 0.7, 0.9],
    "cognitive": [1.0, 1.4, 2.0],
    "social": [1.0, 1.4, 2.0],
}
pso_rastrigin_results = grid_search(
    make_population_runner(run_pso, rastrigin, pso_fixed, dimension=3, domain=(-5.12, 5.12)),
    pso_grid,
)
show_top(pso_rastrigin_results)

1. {'inertia': 0.7, 'cognitive': 1.4, 'social': 1.0, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
2. {'inertia': 0.7, 'cognitive': 2.0, 'social': 1.0, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
3. {'inertia': 0.7, 'cognitive': 2.0, 'social': 2.0, 'mean_best_value': 7.616426008401807e-11, 'std_best_value': 6.080114053811693e-11, 'success_rate': 1.0}
4. {'inertia': 0.9, 'cognitive': 1.4, 'social': 1.0, 'mean_best_value': 0.0001625400752220211, 'std_best_value': 7.77288345243483e-05, 'success_rate': 0.0}
5. {'inertia': 0.9, 'cognitive': 1.0, 'social': 1.4, 'mean_best_value': 0.00039510928050034977, 'std_best_value': 0.00037800584047494495, 'success_rate': 0.3333333333333333}


In [31]:
# Busqueda de grilla para PSO en Rosenbrock 3D
pso_rosenbrock_results = grid_search(
    make_population_runner(run_pso, rosenbrock, pso_fixed, dimension=3, domain=(-2, 2)),
    pso_grid,
)
show_top(pso_rosenbrock_results)

1. {'inertia': 0.7, 'cognitive': 1.0, 'social': 1.4, 'mean_best_value': 0.00014365844529307714, 'std_best_value': 0.00011796082587178302, 'success_rate': 0.6666666666666666}
2. {'inertia': 0.7, 'cognitive': 1.0, 'social': 2.0, 'mean_best_value': 0.00034439100163935113, 'std_best_value': 0.000445567853747107, 'success_rate': 0.6666666666666666}
3. {'inertia': 0.7, 'cognitive': 1.4, 'social': 2.0, 'mean_best_value': 0.001161506539157257, 'std_best_value': 0.001323110246435534, 'success_rate': 0.3333333333333333}
4. {'inertia': 0.9, 'cognitive': 1.0, 'social': 1.0, 'mean_best_value': 0.001291296087152503, 'std_best_value': 0.00018449590634293166, 'success_rate': 0.0}
5. {'inertia': 0.7, 'cognitive': 1.0, 'social': 1.0, 'mean_best_value': 0.0014245241330255957, 'std_best_value': 0.00013863435461789904, 'success_rate': 0.0}


# Optimizacion del algoritmo evolutivo

El presupuesto permanece fijo: 20 individuos, 10 hijos por generacion y 998 generaciones. Se exploran cuatro parametros de comportamiento:

- tournament_size: presion de seleccion.
- crossover_alpha: amplitud del cruce BLX-alpha.
- mutation_rate: probabilidad de mutar cada coordenada.
- mutation_sigma: desviacion de la mutacion gaussiana.

elite_fraction=0.5 queda fijo porque modificarlo cambia el numero de hijos por generacion y dejaria de haber exactamente 10000 evaluaciones.

In [5]:
# Busqueda del evolutivo en Rastrigin 3D
evolutionary_fixed = {
    "population_size": 20,
    "generations": 200,

}
evolutionary_grid = {
    "tournament_size": [2, 3, 5],
    "crossover_alpha": [0.0, 0.3, 0.6],
    "mutation_rate": [0.25, 0.5, 0.75],
    "mutation_sigma": [0.05, 0.1, 0.2],
    "elite_fraction": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
}
evolutionary_rastrigin_results = grid_search(
    make_population_runner(run_evolutionary, rastrigin, evolutionary_fixed, dimension=3, domain=(-5.12, 5.12)),
    evolutionary_grid,
)
show_top(evolutionary_rastrigin_results)

1. {'tournament_size': 3, 'crossover_alpha': 0.6, 'mutation_rate': 0.25, 'mutation_sigma': 0.2, 'elite_fraction': 0.4, 'mean_best_value': 0.6633342679138963, 'std_best_value': 0.4690459885559722, 'success_rate': 0.3333333333333333}
2. {'tournament_size': 2, 'crossover_alpha': 0.6, 'mutation_rate': 0.5, 'mutation_sigma': 0.2, 'elite_fraction': 0.2, 'mean_best_value': 0.6682270916279928, 'std_best_value': 0.4624736949389799, 'success_rate': 0.0}
3. {'tournament_size': 2, 'crossover_alpha': 0.0, 'mutation_rate': 0.5, 'mutation_sigma': 0.05, 'elite_fraction': 0.8, 'mean_best_value': 0.9950202551179856, 'std_best_value': 4.570294774505566e-05, 'success_rate': 0.0}
4. {'tournament_size': 3, 'crossover_alpha': 0.0, 'mutation_rate': 0.5, 'mutation_sigma': 0.1, 'elite_fraction': 0.5, 'mean_best_value': 0.9950312945702405, 'std_best_value': 0.8123721080015271, 'success_rate': 0.3333333333333333}
5. {'tournament_size': 3, 'crossover_alpha': 0.0, 'mutation_rate': 0.5, 'mutation_sigma': 0.1, 'elite

In [6]:
# Busqueda del evolutivo en Rosenbrock 3D
evolutionary_rosenbrock_results = grid_search(
    make_population_runner(run_evolutionary, rosenbrock, evolutionary_fixed, dimension=3, domain=(-2, 2)),
    evolutionary_grid,
)
show_top(evolutionary_rosenbrock_results)

1. {'tournament_size': 5, 'crossover_alpha': 0.3, 'mutation_rate': 0.75, 'mutation_sigma': 0.1, 'elite_fraction': 0.2, 'mean_best_value': 0.0277373983439691, 'std_best_value': 0.018393799596528367, 'success_rate': 0.0}
2. {'tournament_size': 5, 'crossover_alpha': 0.3, 'mutation_rate': 0.75, 'mutation_sigma': 0.2, 'elite_fraction': 0.2, 'mean_best_value': 0.03595275702919639, 'std_best_value': 0.017365541047988424, 'success_rate': 0.0}
3. {'tournament_size': 5, 'crossover_alpha': 0.0, 'mutation_rate': 0.75, 'mutation_sigma': 0.05, 'elite_fraction': 0.2, 'mean_best_value': 0.040188038496187635, 'std_best_value': 0.028195542285696165, 'success_rate': 0.0}
4. {'tournament_size': 5, 'crossover_alpha': 0.0, 'mutation_rate': 0.75, 'mutation_sigma': 0.2, 'elite_fraction': 0.2, 'mean_best_value': 0.05428517367098767, 'std_best_value': 0.036409647893253745, 'success_rate': 0.0}
5. {'tournament_size': 2, 'crossover_alpha': 0.6, 'mutation_rate': 0.75, 'mutation_sigma': 0.05, 'elite_fraction': 0.2,

# Optimizacion por evolucion diferencial

El presupuesto permanece fijo: 40 individuos y 999 generaciones, es decir, 40000 evaluaciones de f. Solo se exploran sus dos hiperparametros de comportamiento:

- f: amplitud de la mutacion diferencial.
- cr: probabilidad de heredar cada coordenada del vector mutante.

In [34]:
# Busqueda de grilla para DE en Rastrigin 3D
de_fixed = {"population_size": 20, "generations": 499}
de_grid = {
    "f": [0.5, 0.7, 0.9],
    "cr": [0.5, 0.7, 0.9],
}
de_rastrigin_results = grid_search(
    make_population_runner(run_differential_evolution, rastrigin, de_fixed, dimension=3, domain=(-5.12, 5.12)),
    de_grid,
)
show_top(de_rastrigin_results)

1. {'f': 0.5, 'cr': 0.5, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
2. {'f': 0.5, 'cr': 0.7, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
3. {'f': 0.5, 'cr': 0.9, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
4. {'f': 0.7, 'cr': 0.5, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
5. {'f': 0.7, 'cr': 0.7, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}


In [35]:
# Busqueda de grilla para DE en Rosenbrock 3D
de_rosenbrock_results = grid_search(
    make_population_runner(run_differential_evolution, rosenbrock, de_fixed, dimension=3, domain=(-2, 2)),
    de_grid,
)
show_top(de_rosenbrock_results)

1. {'f': 0.7, 'cr': 0.9, 'mean_best_value': 0.0, 'std_best_value': 0.0, 'success_rate': 1.0}
2. {'f': 0.9, 'cr': 0.9, 'mean_best_value': 1.653649672569546e-28, 'std_best_value': 2.111481322910258e-28, 'success_rate': 1.0}
3. {'f': 0.7, 'cr': 0.7, 'mean_best_value': 2.0916208081872624e-19, 'std_best_value': 2.928979395310846e-19, 'success_rate': 1.0}
4. {'f': 0.9, 'cr': 0.7, 'mean_best_value': 6.212151624877967e-15, 'std_best_value': 6.935527900256259e-15, 'success_rate': 1.0}
5. {'f': 0.7, 'cr': 0.5, 'mean_best_value': 2.062267245218527e-09, 'std_best_value': 1.5354335204081228e-09, 'success_rate': 1.0}


# Mejores configuraciones encontradas

Las siguientes configuraciones son las mejores de la grilla exploratoria en 3D, usando 3 semillas (42, 43 y 44), 10000 evaluaciones de la funcion por configuracion y ordenamiento por mean_best_value. En empates se conserva la primera combinacion de la grilla.

| Metodo | Funcion | Parametros fijos | Mejores hiperparametros | Media de f | Desv. est. | Tasa de exito |
|---|---|---|---|---:|---:|---:|
| GD | Rosenbrock | 2500 iteraciones exploratorias | learning_rate=0.0005 | 0.092377 | 0.022986 | 0/3 |
| PSO | Rastrigin | 20 particulas, 499 iteraciones | inertia=0.7, cognitive=1.4, social=1.0 | 0.000000 | 0.000000 | 3/3 |
| PSO | Rosenbrock | 20 particulas, 499 iteraciones | inertia=0.7, cognitive=1.0, social=1.4 | 0.000144 | 0.000118 | 2/3 |
| Evolutivo | Rastrigin | 20 individuos, 998 generaciones, elite_fraction=0.5 | tournament_size=3, crossover_alpha=0.0, mutation_rate=0.5, mutation_sigma=0.1 | 0.994980 | 0.812357 | 1/3 |
| Evolutivo | Rosenbrock | 20 individuos, 998 generaciones, elite_fraction=0.5 | tournament_size=2, crossover_alpha=0.6, mutation_rate=0.75, mutation_sigma=0.2 | 0.001711 | 0.000427 | 0/3 |
| DE | Rastrigin | 20 individuos, 499 generaciones | f=0.5, cr=0.5 | 0.000000 | 0.000000 | 3/3 |
| DE | Rosenbrock | 20 individuos, 499 generaciones | f=0.7, cr=0.9 | 0.000000 | 0.000000 | 3/3 |

Estas configuraciones son resultados de una fase exploratoria con presupuesto reducido; deben validarse posteriormente con las 30 corridas y el presupuesto completo de configs/part1.yaml antes de usarlas como conclusion definitiva. GD se mantiene como referencia para Rosenbrock, igual que en el diseño original del cuaderno.

# Optimización de el TSP

# Optimización de hiperparámetros para TSP con GA y ACO

Se parte de una ampolia selección de parámetros, las generaciones/iteraciones no forman parte de la grilla: permanecen fijas para que todas las configuraciones tengan el mismo presupuesto exploratorio.

Para reducir el coste respecto a las corridas reales se usan **30 individuos y 60 generaciones** en GA, y **10 hormigas y 20 iteraciones** en ACO. Estos valores son iguales para toda la búsqueda y no se interpretan como hiperparámetros optimizados. Se evalúan las configuraciones con las semillas 42, 43 y 44; la métrica principal es el coste total medio del mejor tour.


In [7]:
# Grid Search reproducible para GA y ACO en el TSP
from itertools import product

cost_matrix_tsp = compute_total_cost_matrix(hourly_rate=25.0)
TSP_SEEDS = (42, 43, 44)
GA_FIXED = {"population_size": 30, "generations": 60, "record_evolution": False}
ACO_FIXED = {"n_ants": 10, "iterations": 20, "record_evolution": False}

# Los parámetros de presupuesto se mantienen fuera de estas grillas.
GA_GRID = {
    "mutation_rate": [0.10, 0.25, 0.50],
    "tournament_size": [2, 4, 6],
    "elite_size": [1, 2, 4],
}
ACO_GRID = {
    "alpha": [0.5, 1.0, 2.0],
    "beta": [1.5, 2.5, 4.0],
    "rho": [0.05, 0.10, 0.25],
    "q": [50.0, 100.0, 200.0],
}

def grid_search_tsp(algorithm, fixed_parameters, parameter_grid, cost_matrix=cost_matrix_tsp, seeds=TSP_SEEDS):
    keys = tuple(parameter_grid)
    rows = []
    for values in product(*(parameter_grid[key] for key in keys)):
        parameters = dict(zip(keys, values))
        results = []
        for seed in seeds:
            results.append(algorithm(cost_matrix=cost_matrix, seed=seed, **fixed_parameters, **parameters))
        scores = np.asarray([result["best_cost"] for result in results], dtype=float)
        rows.append({**parameters, "mean_best_cost": float(np.mean(scores)), "std_best_cost": float(np.std(scores)), "best_costs": scores.tolist()})
    return sorted(rows, key=lambda row: (row["mean_best_cost"], row["std_best_cost"]))

ga_grid_results = grid_search_tsp(run_genetic_tsp, GA_FIXED, GA_GRID)
aco_grid_results = grid_search_tsp(run_ant_colony_tsp, ACO_FIXED, ACO_GRID)

print("Mejores configuraciones GA:")
for rank, row in enumerate(ga_grid_results[:5], start=1):
    print(f"{rank}. {row}")
print("\nMejores configuraciones ACO:")
for rank, row in enumerate(aco_grid_results[:5], start=1):
    print(f"{rank}. {row}")


Mejores configuraciones GA:
1. {'mutation_rate': 0.5, 'tournament_size': 6, 'elite_size': 2, 'mean_best_cost': 3605.6697324222223, 'std_best_cost': 87.50863187610977, 'best_costs': [3701.461546266667, 3625.6309347777774, 3489.916716222223]}
2. {'mutation_rate': 0.5, 'tournament_size': 6, 'elite_size': 1, 'mean_best_cost': 3659.3566898592594, 'std_best_cost': 34.5293508611904, 'best_costs': [3695.2108338444436, 3612.7194389111114, 3670.139796822223]}
3. {'mutation_rate': 0.25, 'tournament_size': 6, 'elite_size': 4, 'mean_best_cost': 3819.091458325926, 'std_best_cost': 209.9551424228149, 'best_costs': [4064.1962063333335, 3841.675191555556, 3551.4029770888887]}
4. {'mutation_rate': 0.25, 'tournament_size': 6, 'elite_size': 1, 'mean_best_cost': 3833.389943451852, 'std_best_cost': 152.125980200268, 'best_costs': [3900.3894820666665, 3976.940356711112, 3622.8399915777786]}
5. {'mutation_rate': 0.5, 'tournament_size': 4, 'elite_size': 1, 'mean_best_cost': 3848.663993518519, 'std_best_cost': 

# Resultados del Grid Search TSP

Los resultados se ordenan por el menor coste medio (mean_best_cost) y, en caso de empate, por menor desviación estándar. Son resultados exploratorios: deben validarse con las configuraciones ganadoras y el presupuesto completo de los archivos YAML antes de incorporarlas a las corridas finales.

Las búsquedas ejecutadas por la celda anterior quedan disponibles en ga_grid_results y aco_grid_results. Para consultar la mejor configuración:

```python
a = ga_grid_results[0]
b = aco_grid_results[0]
print("GA:", a)
print("ACO:", b)
```


# Resultados obtenidos del Grid Search TSP

La búsqueda ejecutó **27 configuraciones de GA** y **81 configuraciones de ACO**, cada una con las semillas 42, 43 y 44. El presupuesto permaneció fijo dentro de cada algoritmo: GA usó 30 individuos y 60 generaciones; ACO usó 10 hormigas y 20 iteraciones.

| Algoritmo | Mejores hiperparámetros | Coste medio (€) | Desv. estándar (€) | Costes por semilla (€) |
|---|---|---:|---:|---|
| GA | mutation_rate=0.50, tournament_size=6, elite_size=2 | 3605.67 | 87.51 | 3701.46, 3625.63, 3489.92 |
| ACO | alpha=0.50, beta=4.0, rho=0.05, q=200.0 | 2587.48 | 61.11 | 2607.77, 2650.09, 2504.58 |

ACO obtiene el menor coste medio en esta exploración reducida. Esto no constituye una comparación definitiva entre algoritmos, porque GA y ACO tienen representaciones y presupuestos diferentes; las configuraciones ganadoras deben validarse posteriormente con los valores completos de configs/tsp_ga.yaml y configs/tsp_aco.yaml.
